# Spheroid CSG Visualization Walkthrough

This notebook walks through the unit-box spheroid examples using the live pydetran CSG and tracking API. It creates sampled slice SVGs, 3-D tracker projections, and exact analytic conic SVGs for visual comparison.

Start Jupyter from a shell with the local Detran runtime on the path:

```bash
export DETRAN_ROOT=/home/robertsj/Research/detran
export PATH=$DETRAN_ROOT/install/bin:$PATH
export LD_LIBRARY_PATH=$DETRAN_ROOT/install/lib:$LD_LIBRARY_PATH
export PYTHONPATH=$DETRAN_ROOT/install/python:$PYTHONPATH
cd $DETRAN_ROOT/detran-examples/csg
jupyter notebook
```

The `LD_LIBRARY_PATH` setting needs to be present before the Python kernel starts.

In [ ]:
from pathlib import Path
import json
import math
import os
import sys

from IPython.display import SVG, Markdown, display

DETRAN_ROOT = Path(os.environ.get("DETRAN_ROOT", "/home/robertsj/Research/detran")).resolve()
EXAMPLE_DIR = DETRAN_ROOT / "detran-examples" / "csg"
INSTALL_PYTHON = DETRAN_ROOT / "install" / "python"

for path in (EXAMPLE_DIR, INSTALL_PYTHON):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import detran as dt
import spheroid_performance as sph

OUTPUT_DIR = EXAMPLE_DIR / "notebook_outputs" / "spheroid"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Detran root: {DETRAN_ROOT}")
print(f"Notebook outputs: {OUTPUT_DIR}")

## Built-In Cases

Each case is an axis-aligned spheroid in the unit box. The semi-axes are scaled so every extent remains inside the `[0.01, 0.99]` guard band.

In [ ]:
def markdown_table(headers, rows):
    table = ["| " + " | ".join(headers) + " |"]
    table.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        table.append("| " + " | ".join(str(value) for value in row) + " |")
    return "\n".join(table)

case_rows = []
for case in sph.CASES.values():
    case_rows.append([
        case.name,
        tuple(round(v, 4) for v in case.center),
        tuple(round(v, 4) for v in case.semi_axes),
        tuple((round(lo, 4), round(hi, 4)) for lo, hi in case.extents),
        f"{case.exact_volume:.6f}",
    ])

display(Markdown(markdown_table(
    ["case", "center", "semi-axes", "extents", "exact volume"],
    case_rows,
)))

## Centered Sphere: Sampled Slice, Tracks, and Analytic Conics

The first reference case is the sphere centered at `(0.5, 0.5, 0.5)`. The middle `z = 0.5` slice is a circle in the unit square with radius `0.49`.

In [ ]:
case = sph.CASES["centered_sphere"]
case_dir = OUTPUT_DIR / case.name

result = sph.benchmark_case(
    case,
    directions=sph.DIRECTION_SETS["xyz"],
    spacing=0.08,
    slice_cells=80,
    output_dir=case_dir,
)

sph.print_summary(result)

In [ ]:
display(Markdown("### Sampled native CSG slice"))
display(SVG(filename=str(case_dir / f"{case.name}_slice_xy.svg")))

display(Markdown("### Exact analytic slice conic"))
display(SVG(filename=str(case_dir / f"{case.name}_analytic_slice_xy.svg")))

display(Markdown("### 3-D tracker projected onto x-y"))
display(SVG(filename=str(case_dir / f"{case.name}_tracks_xy.svg")))

display(Markdown("### Exact orthographic projection footprint"))
display(SVG(filename=str(case_dir / f"{case.name}_analytic_projection_xy.svg")))

## Hand-Check the Geometry

The centerline through the sphere has three path lengths: box shell, spheroid chord, box shell. Those are `0.01`, `0.98`, and `0.01` along each coordinate axis.

In [ ]:
model = case.model()
centerline = model.track(vectors=sph.DIRECTION_SETS["x"], spacing=10.0, minimum_tracks=1)

display(Markdown(markdown_table(
    ["segment", "region id", "length", "expected length"],
    [
        [i, int(region), f"{length:.8f}", f"{expected:.8f}"]
        for i, (region, length, expected) in enumerate(zip(
            centerline.segments.region_id,
            centerline.segments.length,
            case.centerline_lengths(0),
        ))
    ],
)))

## Oblong and Off-Center Cases

These smaller smoke runs are useful for checking coordinate indexing and visual conventions. The volume error is the track-estimated spheroid volume minus the exact analytic volume.

In [ ]:
summary_rows = []
for name in ("centered_sphere", "oblong_x", "offcenter_x", "offcenter_y", "offcenter_z"):
    c = sph.CASES[name]
    r = sph.benchmark_case(
        c,
        directions=sph.DIRECTION_SETS["xyz"],
        spacing=0.12,
        slice_cells=48,
    )
    summary_rows.append([
        name,
        r["tracks"],
        r["segments"],
        f"{r['exact_spheroid_volume']:.6f}",
        f"{r['tracked_region_volumes'].get(sph.SPHEROID_REGION, 0.0):.6f}",
        f"{r['spheroid_volume_error']:.3e}",
        f"{r['timings']['track_s']:.4f}",
    ])

display(Markdown(markdown_table(
    ["case", "tracks", "segments", "exact volume", "tracked volume", "error", "track seconds"],
    summary_rows,
)))

## Native Quadratic Feature Coordinates

The analytic feature API reports projected conics in real coordinates. For an axis-aligned spheroid, the projection onto `xy`, `xz`, and `yz` has the same center components and the corresponding semi-axes.

In [ ]:
feature_model = sph.CASES["oblong_x"].model()
feature_rows = []
for plane in ("xy", "xz", "yz"):
    conic = feature_model.quadratic_projections(plane)[0]
    feature_rows.append([
        plane,
        tuple(conic.axis_labels),
        tuple(round(v, 6) for v in conic.center),
        tuple(round(v, 6) for v in conic.axes),
        tuple(round(v, 6) for v in conic.bounds),
    ])

display(Markdown(markdown_table(
    ["plane", "axes", "center", "semi-axes", "bounds"],
    feature_rows,
)))

## Local Playground Knobs

Change these values to exercise local hardware. Smaller spacing increases the number of generated 3-D tracks; larger `slice_cells` increases sampled slice resolution.

In [ ]:
case_name = "oblong_x"
direction_name = "diagonal"
spacing = 0.06
slice_cells = 96

play_case = sph.CASES[case_name]
play_dir = OUTPUT_DIR / f"play_{case_name}_{direction_name}"
play_result = sph.benchmark_case(
    play_case,
    directions=sph.DIRECTION_SETS[direction_name],
    spacing=spacing,
    slice_cells=slice_cells,
    output_dir=play_dir,
)

sph.print_summary(play_result)
display(SVG(filename=str(play_dir / f"{play_case.name}_tracks_xy.svg")))